In [2]:

# Uppdaterar ändringar i src utan att behöva starta om kernel:

%load_ext autoreload 
%autoreload 2
%matplotlib inline


import sys
import os
import time

# Check kernel
print(f'path: {sys.executable}')


from typing import Tuple
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

from heston_project.models.DDN import *
from heston_project.utils import *


# Device Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Reproducibility 
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if device.type == 'cuda':
    torch.cuda.manual_seed(SEED)

print("Setup Complete.")

path: /Users/manswestman/Kandidatarbete/venv/bin/python
Using device: cpu
Setup Complete.


In [3]:
df = pd.read_csv('../../data/dataset_100K_nofeller.csv')
# df = pd.read_csv(project_root/'data'/'dataset_100K_nofeller.csv')

# Shuffle with a fixed random state for reproducibility
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df_scaler = df.copy()

#Scale theta & v0 (Ska dessa behållas i vanlig form också för att inte scalea om i output)
df_scaler[['theta', 'v0']] = np.sqrt(df_scaler[['theta', 'v0']])
sqrt_copy = df_scaler[['theta', 'v0']].copy()

inputs = df_scaler[['kappa', 'theta', 'v0', 'rho', 'sigma', 'r', 'lm', 'tau']]
labels = df_scaler[['P_hat', 'diff wrt lm', 'diff wrt r', 'diff wrt tau', 'diff wrt theta', 'diff wrt sigma', 'diff wrt rho', 'diff wrt kappa', 'diff wrt v0']]

'''
Parameter bounds (actual dataset is in utils.py for consistency across notebooks)
    lm:    [-2.0, 2.0]
    r:     [-0.01, 0.10]
    tau:   [0.05, 20.0]
    theta: [0.0, 1.0]
    sigma: [0.1, 2.0]
    rho:   [-0.90, 0.0]
    kappa: [0.005, 3.0]
    v0:    [0.0, 1.0]
'''


# Skalar inputs till [-1,1]
inputs = scale_inputs(inputs, zero_centered=True)

# Skalar price till [0,1]
labels['P_hat'] = scale_price(labels['P_hat'], zero_centered=False)


# Skalar greek targets till dP_tilde/dx (där x=inputs skalat till [-1,1]) genom kedjeregeln
def scale_greek_target(greek, xmin, xmax, p_max, p_min):
    return ((xmax - xmin) * greek) / (2 * (p_max - p_min))

pmin = labels['P_hat'].min()
pmax = labels['P_hat'].max()

for k, (kmin, kmax) in param_bounds.items():
    greek_label = 'diff wrt ' + str(k)

    if k in ["theta", "v0"]: #En extra term i kedjeregeln krävs för att ta hänsyn till sqrt-transformationen
        labels[greek_label] = scale_greek_target(labels[greek_label], kmin, kmax, pmin, pmax) * (2 * sqrt_copy[k])
    else:
        labels[greek_label] = scale_greek_target(labels[greek_label], kmin, kmax, pmin, pmax)


#Split data
X_train, X_val, y_train, y_val = train_test_split(inputs, labels, test_size=0.1, random_state=42)

train_dataset = TensorDataset(
    torch.tensor(X_train.values, dtype=torch.float32),
    torch.tensor(y_train.values, dtype=torch.float32)
    )
val_dataset = TensorDataset(
    torch.tensor(X_val.values, dtype=torch.float32),
    torch.tensor(y_val.values, dtype=torch.float32)
    )

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2048)


**Training & Validation Steps**

In [4]:
# Bestämmer vilka greeks vi plockar ut för pred_grads och y_grads som ska användas i lossfunktionen.
relevant_greeks = ['kappa', 'v0', 'rho', 'sigma', 'theta']

# y och full_grad bygger på dataset med icke-matchande kolumnordning     
input_columns = list(inputs.columns)
output_columns = list(labels.columns)

index_in_input = torch.as_tensor([input_columns.index(k) for k in relevant_greeks], device=device)
index_in_output = torch.as_tensor([output_columns.index(f'diff wrt {k}') for k in relevant_greeks], device=device)

In [5]:
def train_step(model, criterion, batch: tuple, optimizer, weights):
        
        x, y = batch

        x.requires_grad_(True) # "håll koll på gradienter"

        optimizer.zero_grad()

        # Price loss
        pred_price = model(x)
        y_price = y[:, 0:1]
        l_p = criterion(pred_price, y_price)


        # Compute gradients (Greeks)
        full_grads = torch.autograd.grad(
            outputs=pred_price,
            inputs=x,
            grad_outputs=torch.ones_like(pred_price),
            create_graph=True, # OBS detta är inte för att få greeks utan för att beräkna gradienten av greeksen för optimeraren!!
            # retain_graph=True, är sant automatiskt
            only_inputs=True
        )[0]

        # Greek loss
        pred_grads = full_grads[:, index_in_input]
        y_grads = y[:, index_in_output]
        l_g = criterion(pred_grads, y_grads)
        

        # Laplace loss
        laplacian_penalty = 0.0
    
        # Loop over all input dimensions to get the unmixed second derivatives
        for i in range(x.shape[1]):
            # Isolate the first derivative for dimension i
            g_i = full_grads[:, i].unsqueeze(1)
            
            # Differentiate g_i with respect to all inputs
            h_row = torch.autograd.grad(
                outputs=g_i,
                inputs=x,
                grad_outputs=torch.ones_like(g_i),
                create_graph=True,
                retain_graph=True
            )[0]
            
            # Extract the pure second derivative (e.g., d^2P / dx_i^2)
            second_deriv = h_row[:, i]
            laplacian_penalty += (second_deriv ** 2).mean()
            

        # Get loss weights
        lambda_p = weights['lambda_p']  
        lambda_g = weights['lambda_g']
        lambda_reg = weights['lambda_reg']

        
        # Loss function
        loss = (lambda_p * l_p) + (lambda_g * l_g) + (lambda_reg * laplacian_penalty)

        # Compute loss gradients and update parameters
        loss.backward()
        optimizer.step()

        return {"loss": loss.item(), "l_p": l_p.item(), "l_g": l_g.item(), "loss_laplace": laplacian_penalty}

In [6]:
def val_step(model, batch: Tuple, criterion, weights):

    x, y = batch

    x.requires_grad_(True)

    # Price loss
    pred_price = model(x)
    y_price = y[:, 0:1]
    l_p = criterion(pred_price, y_price)
    

    # Compute gradients (Greeks)
    full_grads = torch.autograd.grad(
            outputs=pred_price,
            inputs=x,
            grad_outputs=torch.ones_like(pred_price),
            create_graph=False, # Behövs ej vid test/validering (se komentar för training_step)
            retain_graph=False
        )[0]
        
    # Greek loss
    pred_grads = full_grads[:, index_in_input]
    y_grads = y[:, index_in_output] 
    l_g = criterion(pred_grads, y_grads)
    

    # Get loss weights
    lambda_p = weights['lambda_p']  
    lambda_g = weights['lambda_g']


    # Loss function (Laplacian loss not included in validation)
    loss = (lambda_p * l_p) + (lambda_g * l_g)

    return {"loss": loss.item(), "l_p": l_p.item(), "l_g": l_g.item()}

**HYPERPARAMETERS**

In [7]:
# Loss weights
lambda_p = 10
lambda_g = 1
lambda_reg = 5e-5

# Model parameters 
nr_input_dim = 8
nr_hidden_layers = 7
nr_neurons = 250

# Optimizer settings
learning_rate = 3e-4
weight_decay = 1e-5
scheduler_patience = 7
scheduler_factor = 0.5

# Training loop
EPOCHS = 500
patience_early_stopping = 25

In [8]:
# initialisera modellen till device
model = DDN(input_dim=nr_input_dim, hidden_layers=nr_hidden_layers, neurons=nr_neurons).to(device)

# Loss + Optimizer
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=scheduler_patience, factor=scheduler_factor) # bestämer hur och när vi ska ändra lr 

**TRAINING LOOP**

In [ ]:
# Path for saving best model state
import heston_project
PKG_DIR = Path(heston_project.__file__).resolve().parent
SAVE_DIR = PKG_DIR / "models" / "saved"
out_path = SAVE_DIR / "DDN_P_Laplace_best.pth"

# Tracker for early stopping
counter = 0

# Loss dicts
train_losses = {'total_train_loss': [], 'lp_train': [], 'lg_train': [], 'lap_train': []}
val_losses = {'total_val_loss': [], 'lp_val': [], 'lg_val': [], 'lap_val': []}

best_val_loss = 1000

#Loss weights
weights_dict = {'lambda_p': lambda_p, 'lambda_g': lambda_g, 'lambda_reg': lambda_reg}

print(f'Starting Training on {device}...')
start_time = time.time()        ##########

for epoch in range(EPOCHS):
    model.train()
    running_tot_loss, running_lp_loss, running_lg_loss, running_lap_loss = 0.0, 0.0, 0.0, 0.0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        losses = train_step(model, criterion, batch = (batch_X, batch_y), optimizer = optimizer, weights = weights_dict)

        running_tot_loss += losses['loss']
        running_lp_loss += losses['l_p']
        running_lg_loss += losses['l_g']
        running_lap_loss += losses['loss_laplace']
    
    avg_train_total_loss = running_tot_loss/len(train_loader)
    avg_train_lp_loss = running_lp_loss/len(train_loader)
    avg_train_lg_loss = running_lg_loss/len(train_loader)
    avg_train_lap_loss = running_lap_loss/len(train_loader)

    train_losses['total_train_loss'].append(avg_train_total_loss)
    train_losses['lp_train'].append(avg_train_lp_loss)
    train_losses['lg_train'].append(avg_train_lg_loss)
    train_losses['lap_train'].append(avg_train_lap_loss)

    model.eval()
    running_tot_loss, running_lp_loss, running_lg_loss, running_lap_loss = 0.0, 0.0, 0.0, 0.0

    # OBS ANVÄND ABSOLUT INTE TORCH NO GRAD
    for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)


            losses = val_step(model, batch = (batch_X, batch_y), criterion = criterion, weights = weights_dict)

            running_tot_loss += losses['loss']
            running_lp_loss += losses['l_p']
            running_lg_loss += losses['l_g']
    
    avg_val_total_loss = running_tot_loss/len(val_loader)
    avg_val_lp_loss = running_lp_loss/len(val_loader)
    avg_val_lg_loss = running_lg_loss/len(val_loader)


    val_losses['total_val_loss'].append(avg_val_total_loss)
    val_losses['lp_val'].append(avg_val_lp_loss)
    val_losses['lg_val'].append(avg_val_lg_loss)

    # Uppdatera Learning Rate
    scheduler.step(avg_val_total_loss)

    if avg_val_total_loss < best_val_loss:
        best_val_loss = avg_val_total_loss
        best_loss_price = avg_val_lp_loss
        best_loss_greeks = avg_val_lg_loss
        torch.save(model.state_dict(), out_path) # Save the best version
        counter = 0 # Återställ!!!
    else:
        counter += 1
        if counter >= patience_early_stopping:
            print(f"Early stopping triggered at epoch {epoch}")
            break
    
    if (epoch + 1) % 5 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch [{epoch+1:3d}/{EPOCHS}] | LR: {current_lr:.2e}")
        print(f"  Train -> Tot: {avg_train_total_loss:.4e} | Price: {avg_train_lp_loss:.4e} | Greeks: {avg_train_lg_loss:.4e}")
        print(f"  Val   -> Tot: {avg_val_total_loss:.4e} | Price: {avg_val_lp_loss:.4e} | Greeks: {avg_val_lg_loss:.4e}")
        print(f"  TRAIN LAPLACE: {avg_train_lap_loss:.4e}")
        print("-" * 70)

end_time = time.time() 
elapsed_time = end_time - start_time
print("*" * 70)
print(f"Best Val Loss -> Tot: {best_val_loss:.4e} | Price: {best_loss_price:.4e} | Greeks: {best_loss_greeks:.4e}")
print("*" * 70)
print(f'Training time: {elapsed_time:.2f} seconds')